In [21]:
import numpy as np
import statsmodels.api as sm
from sklearn.decomposition import PCA
from concurrent.futures import ProcessPoolExecutor, as_completed
import itertools
import logging
from one.api import ONE
from brainbox.io.one import SessionLoader
from brainwidemap import bwm_query, load_good_units, load_trials_and_mask, bwm_units
from collections import defaultdict
import pandas as pd
from manifold.decoding.functions.utils import check_config_decoding
import numpy as np
import pickle as pkl
from manifold.decoding.functions import nulldistributions
from communication_subspace.ibl_communication.utils import load_widefield_epoch
from tqdm import tqdm
from iblatlas.atlas import AllenAtlas
from iblatlas.regions import BrainRegions
from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score, balanced_accuracy_score
import warnings
from manifold.utils import get_trial_masks
from scipy.stats import pointbiserialr
from manifold.widefield_ppi import beryl_mapping,aggregate_by_parent
from matplotlib import pyplot as plt

with open("../data/processed/significant_stims_choice.pkl",'rb') as f:
    significant_preloads = pkl.load(f)


config = check_config_decoding()

In [32]:
from manifold.plot_spatial_alignment import plot_alignment

In [33]:
from glob import glob

In [34]:
files = glob('../data/generated/wifi/spatial_alignment/*.pkl')

In [41]:
df = pd.read_csv('../data/generated/wifi/spatial_alignment/batch_spatial_alignment_stats.csv')

In [46]:
df.sort_values('mean_acc_stim_incorrect')

,mean_corr_correct,mean_corr_incorrect,t_stat_corr_delta,p_val_corr_delta,mean_acc_stim_incorrect,mean_acc_choice_incorrect,t_stat_acc_delta,p_val_acc_delta,n_sessions,stim,choice
35,0.177353,0.102087,2.850559,7.015091e-03,0.448655,0.358622,6.160604,3.428301e-07,39,MO,AUD
32,0.240111,0.129370,3.987585,2.842092e-04,0.449668,0.275977,13.333364,4.147989e-16,40,MO,VIS
26,0.227407,0.126145,4.001369,2.727384e-04,0.449668,0.327905,6.882806,3.105239e-08,40,MO,PTLp
25,0.233989,0.136689,4.046693,2.381105e-04,0.449668,0.246818,14.410804,3.299699e-17,40,MO,RSP
33,0.226781,0.090812,5.707475,1.876344e-06,0.460124,0.324712,7.319967,1.483364e-08,36,RSP,PTLp
22,0.237713,0.097646,5.345824,5.628432e-06,0.460124,0.246179,13.509436,1.897411e-15,36,RSP,RSP
2,0.177096,0.080879,4.905139,1.318116e-05,0.475166,0.361832,8.305467,1.475391e-10,45,SS,AUD
8,0.165887,0.089187,4.215053,1.221087e-04,0.475166,0.408478,4.112756,1.683247e-04,45,SS,TEa
37,0.161692,0.064330,5.705857,9.115134e-07,0.475427,0.377107,5.973076,3.696529e-07,45,SS,PL
0,0.232191,0.112857,6.675807,3.087539e-08,0.475688,0.325076,9.944687,6.185508e-13,46,SS,PTLp


In [47]:
with open(files[0], 'rb') as f:
    ss = pkl.load(f)


In [49]:
ssdf = pd.DataFrame(ss)

In [52]:
ssdf

,region,epoch,cv_accuracy,cv_balanced_accuracy,correct_trial_indices,correct_logits,correct_true_labels,incorrect_trial_indices,incorrect_logits,incorrect_true_labels
0,MOB,stim,0.572464,0.571970,"[1, 2, 3, 4, 5, 7, 9, 11, 15, 16, 17, 18, 25, ...","[-0.8717124633827573, 2.943167058946075, 0.197...","[1.0, 1.0, -1.0, -1.0, 1.0, -1.0, 1.0, 1.0, -1...","[0, 8, 20, 22, 23, 27, 31, 33, 34, 36, 37, 44,...","[-3.9266221958478433, 2.1062361098438385, 0.06...","[-1.0, 1.0, 1.0, -1.0, -1.0, -1.0, 1.0, 1.0, 1..."
1,MO,stim,0.608696,0.605114,"[1, 2, 3, 4, 5, 7, 9, 11, 15, 16, 17, 18, 25, ...","[14.4903535108617, 2.7980466123830197, -17.582...","[1.0, 1.0, -1.0, -1.0, 1.0, -1.0, 1.0, 1.0, -1...","[0, 8, 20, 22, 23, 27, 31, 33, 34, 36, 37, 44,...","[-3.124578008730907, -3.1877960923903306, -6.2...","[-1.0, 1.0, 1.0, -1.0, -1.0, -1.0, 1.0, 1.0, 1..."
2,SS,stim,0.565217,0.564078,"[1, 2, 3, 4, 5, 7, 9, 11, 15, 16, 17, 18, 25, ...","[-3.3158119876556187, -0.4084847032919336, 1.1...","[1.0, 1.0, -1.0, -1.0, 1.0, -1.0, 1.0, 1.0, -1...","[0, 8, 20, 22, 23, 27, 31, 33, 34, 36, 37, 44,...","[-8.027830240139263, 3.401121824614416, -5.099...","[-1.0, 1.0, 1.0, -1.0, -1.0, -1.0, 1.0, 1.0, 1..."
3,PL,stim,0.536232,0.531881,"[1, 2, 3, 4, 5, 7, 9, 11, 15, 16, 17, 18, 25, ...","[0.31288031873399447, 0.9848641301039728, -0.0...","[1.0, 1.0, -1.0, -1.0, 1.0, -1.0, 1.0, 1.0, -1...","[0, 8, 20, 22, 23, 27, 31, 33, 34, 36, 37, 44,...","[-1.1842214147827033, 0.9324576656452059, 0.03...","[-1.0, 1.0, 1.0, -1.0, -1.0, -1.0, 1.0, 1.0, 1..."
4,RSP,stim,0.568841,0.567235,"[1, 2, 3, 4, 5, 7, 9, 11, 15, 16, 17, 18, 25, ...","[-1.2157530545863178, 0.47627755913211167, -0....","[1.0, 1.0, -1.0, -1.0, 1.0, -1.0, 1.0, 1.0, -1...","[0, 8, 20, 22, 23, 27, 31, 33, 34, 36, 37, 44,...","[-2.6824297168774653, -0.17551990897279982, -0...","[-1.0, 1.0, 1.0, -1.0, -1.0, -1.0, 1.0, 1.0, 1..."
5,TEa,stim,0.634058,0.632891,"[1, 2, 3, 4, 5, 7, 9, 11, 15, 16, 17, 18, 25, ...","[-0.35866993703141736, -0.40866544117286735, -...","[1.0, 1.0, -1.0, -1.0, 1.0, -1.0, 1.0, 1.0, -1...","[0, 8, 20, 22, 23, 27, 31, 33, 34, 36, 37, 44,...","[-0.8704123544603373, -0.6634413573745201, -0....","[-1.0, 1.0, 1.0, -1.0, -1.0, -1.0, 1.0, 1.0, 1..."
6,AUD,stim,0.623188,0.619949,"[1, 2, 3, 4, 5, 7, 9, 11, 15, 16, 17, 18, 25, ...","[0.22792852603719063, -0.4941084838186096, -0....","[1.0, 1.0, -1.0, -1.0, 1.0, -1.0, 1.0, 1.0, -1...","[0, 8, 20, 22, 23, 27, 31, 33, 34, 36, 37, 44,...","[-0.9944573962498211, 0.13091497590647191, 0.1...","[-1.0, 1.0, 1.0, -1.0, -1.0, -1.0, 1.0, 1.0, 1..."
7,VIS,stim,0.688406,0.687816,"[1, 2, 3, 4, 5, 7, 9, 11, 15, 16, 17, 18, 25, ...","[-0.128095381470982, -1.2427588125330762, -1.6...","[1.0, 1.0, -1.0, -1.0, 1.0, -1.0, 1.0, 1.0, -1...","[0, 8, 20, 22, 23, 27, 31, 33, 34, 36, 37, 44,...","[-2.972212205147403, 1.6325484859897106, -0.24...","[-1.0, 1.0, 1.0, -1.0, -1.0, -1.0, 1.0, 1.0, 1..."
8,PTLp,stim,0.634058,0.632260,"[1, 2, 3, 4, 5, 7, 9, 11, 15, 16, 17, 18, 25, ...","[-0.3699409901861888, -1.6374277368648458, -0....","[1.0, 1.0, -1.0, -1.0, 1.0, -1.0, 1.0, 1.0, -1...","[0, 8, 20, 22, 23, 27, 31, 33, 34, 36, 37, 44,...","[-0.4901068093112892, 1.5078800383029043, 1.29...","[-1.0, 1.0, 1.0, -1.0, -1.0, -1.0, 1.0, 1.0, 1..."
9,MOB,choice,0.768116,0.768624,"[1, 2, 3, 4, 5, 7, 9, 11, 15, 16, 17, 18, 25, ...","[5.385005348329091, 3.551809804038507, -2.9457...","[1.0, 1.0, -1.0, -1.0, 1.0, -1.0, 1.0, 1.0, -1...","[0, 8, 20, 22, 23, 27, 31, 33, 34, 36, 37, 44,...","[-3.1102791634264535, -4.23013976939268, 0.532...","[-1.0, 1.0, 1.0, -1.0, -1.0, -1.0, 1.0, 1.0, 1..."
